# Latent Dynamics Workbench

Interactive notebook covering the same two workflows as the CLI (`extract` and `analyze`).
All logic lives in `src/latent_dynamics/` — this notebook only configures and orchestrates.

**Sections**
1. [Config](#1-config) — pick model, dataset, layer, pooling, direction method
2. [Extract](#2-extract) — load data + model, pull hidden-state trajectories, optionally save
3. [Analyze](#3-analyze) — linear probe, LAT scan, trust-region drift
4. [Persist / Hub](#4-persist--hub) — save to disk and optionally push to HuggingFace Hub

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch

from latent_dynamics.activations import build_feature_matrix, extract_multi_layer_trajectories
from latent_dynamics.analysis import drift_curve, fit_trust_region, lat_scan, make_direction, train_linear_probe
from latent_dynamics.config import DATASET_REGISTRY, MODEL_REGISTRY, RunConfig
from latent_dynamics.data import load_examples, prepare_text_and_labels
from latent_dynamics.hub import activation_subpath, load_activations, push_to_hub, save_activations
from latent_dynamics.models import load_model_and_tokenizer, resolve_device
from latent_dynamics.viz import plot_drift_curves, plot_lat_scans

torch.set_grad_enabled(False)
np.set_printoptions(suppress=True, precision=4)

print("Available models:  ", list(MODEL_REGISTRY))
print("Available datasets:", list(DATASET_REGISTRY))

Available models:   ['qwen3_8b', 'llama_3_1_8b', 'gemma3_4b', 'gemma3_270m']
Available datasets: ['toy_contrastive', 'wildjailbreak', 'xstest']


## 1 — Config

Edit `CFG` and `LAYER_INDICES` here. Re-run the rest of the notebook after changing anything.

In [ ]:
CFG = RunConfig(
    # ── model / data ──────────────────────────────────────────────
    model_key   = "gemma3_4b",       # qwen3_8b | llama_3_1_8b | gemma3_4b
    dataset_key = "toy_contrastive", # toy_contrastive | wildjailbreak | xstest
    split       = "train",
    max_samples = 120,
    max_input_tokens  = 256,

    # ── representation ────────────────────────────────────────────
    layer_idx        = 5,     # primary layer (used for probe/scan when loading single layer)
    pooling          = "last", # last | mean | max_norm
    direction_method = "probe_weight", # probe_weight | mean_diff | pca

    # ── splits ────────────────────────────────────────────────────
    test_size    = 0.25,
    calib_size   = 0.25,
    random_state = 7,

    # ── generation (optional) ─────────────────────────────────────
    use_generate               = False,
    max_new_tokens             = 24,
    include_prompt_in_trajectory = True,
)

# Extract hidden states for all of these layers in a single forward pass.
LAYER_INDICES: list[int] = [CFG.layer_idx]
# Example multi-layer: LAYER_INDICES = [3, 5, 10, 15, 20]

# Output root for saved activations (mirrors CLI --output flag).
ACTIVATIONS_ROOT = Path("./activations")

# Set to a HuggingFace repo id (e.g. "username/my-repo") to push, or None to skip.
PUSH_TO_HUB: str | None = None

CFG.device = resolve_device(CFG.device)
print("Device:", CFG.device)

## 2 — Extract

Load the dataset and model, then extract hidden-state trajectories (one forward pass per example, all requested layers collected at once).

In [ ]:
dataset, spec = load_examples(CFG.dataset_key, CFG.split, CFG.max_samples)
print(f"Rows: {len(dataset)}  |  columns: {dataset.column_names}")
for row in list(dataset)[:2]:
    print(" ", {k: row[k] for k in dataset.column_names})

texts, labels = prepare_text_and_labels(
    dataset,
    text_field  = spec.text_field,
    label_field = spec.label_field,
    label_fn    = spec.label_fn,
)
print(f"\n{len(texts)} examples — labels: {np.unique(labels, return_counts=True) if labels is not None else 'none'}")

In [ ]:
model, tokenizer = load_model_and_tokenizer(CFG.model_key, CFG.device)
print(f"Loaded {MODEL_REGISTRY[CFG.model_key]['hf_id']} on {CFG.device}")

In [ ]:
per_layer, token_texts = extract_multi_layer_trajectories(
    model, tokenizer, texts,
    layer_indices = LAYER_INDICES,
    max_input_tokens    = CFG.max_input_tokens,
    device        = CFG.device,
    cfg           = CFG,
)

# Use the primary layer for the rest of the analysis.
trajectories = per_layer[CFG.layer_idx]
print(f"Extracted {len(trajectories)} trajectories across layers {LAYER_INDICES}")
print(f"  Primary layer {CFG.layer_idx} — example shape: {trajectories[0].shape}")

## 3 — Analyze

Train a linear probe, compute a concept direction, run the LAT scan, then fit a spherical trust-region and plot drift curves.

In [ ]:
if labels is None:
    raise ValueError("Labels are required for probe training. Set label_field or label_fn in config.")

X = build_feature_matrix(trajectories, pooling=CFG.pooling)
print("Feature matrix:", X.shape)

probe, (X_train, y_train), (X_calib, y_calib), (X_test, y_test), metrics = train_linear_probe(
    X, labels,
    test_size    = CFG.test_size,
    calib_size   = CFG.calib_size,
    random_state = CFG.random_state,
)
print(f"Accuracy: {metrics['accuracy']:.3f}  |  ROC-AUC: {metrics['roc_auc']:.3f}")
print(metrics["report"])

In [ ]:
direction = make_direction(CFG.direction_method, X_train, y_train, probe=probe)
scans = lat_scan(trajectories, direction)

plot_lat_scans(
    scans, labels,
    title=f"LAT scan — {CFG.model_key} / {CFG.dataset_key} / L{CFG.layer_idx} / {CFG.direction_method}",
).show()

In [ ]:
safe_center, tau = fit_trust_region(X_calib, y_calib, q=0.95)
print(f"Trust-region tau (95th-pct safe distance): {tau:.4f}")

drifts = [drift_curve(traj, safe_center) for traj in trajectories]

plot_drift_curves(
    drifts, labels, tau,
    title=f"Trust-region drift — {CFG.model_key} / {CFG.dataset_key} / L{CFG.layer_idx}",
).show()

test_dists = np.linalg.norm(X_test - safe_center[None, :], axis=1)
accept = test_dists <= tau
print(f"Acceptance rate on test: {accept.mean():.3f}")
print(f"Unsafe accepted: {int(((y_test == 1) & accept).sum())} / {int((y_test == 1).sum())}")

## 4 — Persist / Hub

Save activations to `ACTIVATIONS_ROOT` (mirroring the CLI `--output` layout) and optionally push to a HuggingFace Hub dataset repo.

To reload saved activations instead of re-extracting, use `load_activations(layer_dir)`.

In [ ]:
for li in LAYER_INDICES:
    import dataclasses
    layer_cfg = dataclasses.replace(CFG, layer_idx=li)
    sub = activation_subpath(CFG.dataset_key, CFG.split, CFG.model_key, li)
    layer_dir = ACTIVATIONS_ROOT / sub
    save_activations(layer_dir, per_layer[li], texts, labels, token_texts, layer_cfg)
    print(f"Saved layer {li} → {layer_dir}")

In [ ]:
if PUSH_TO_HUB:
    url = push_to_hub(ACTIVATIONS_ROOT, PUSH_TO_HUB)
    print("Pushed to", url)
else:
    print("Skipping Hub push (set PUSH_TO_HUB to a repo id to enable).")

In [ ]:
# ── Reload from disk (optional) ──────────────────────────────────────────────
# Uncomment and point to any previously saved leaf directory to skip re-extraction.
#
# reload_dir = ACTIVATIONS_ROOT / activation_subpath(
#     CFG.dataset_key, CFG.split, CFG.model_key, CFG.layer_idx
# )
# trajectories, texts, labels, token_texts, cfg_loaded = load_activations(reload_dir)
# print(f"Reloaded {len(trajectories)} trajectories from {reload_dir}")